# Task 3 — Gender name-truth labels

Run All trains **folds 0 and 4 only**, from scratch, using the new name-based labels.
Keep **dropout 0.30**, **grayscale probability 0.10** and the same mild darkening as 04ac.
Only the gender labels change. All images and fold assignments stay the same.

Use a **fresh Colab L4** matching the previous runs. Push this notebook and its source files before running. The completed 04ac grayscale runs, their earlier parents and 04w precision evidence must be on Drive.

## 1. Colab GPU and repository


In [1]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Commit: 854d7703adcae426b584d9161abe963ba80938d3


## 2. Teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. Freeze the labels and check the parents

Use the product name when it contains exactly one explicit gender cue. Keep and flag unclear names. This changes **350 development labels**, including validation labels. The original `data/processed/splits.csv` stays unchanged. Held-out and quarantined rows stay sealed.

Build the small label files locally from the canonical split; no new data ZIP is needed. Save a copy beside the Drive results. The label-file and summary hashes are part of the run configuration, so runs with different labels cannot be reused.

Keep widths `[32, 64, 128, 256]`, **390,181 parameters**, full images, GeM p=3, dropout 0.30, clean fold-training RGB normalization, cross-entropy, AdamW weight decay 0.0001, the G2 learning rate and cosine schedule, batch 128, **30 epochs**, seed **2753**, and the final-epoch checkpoint.

Training augmentation stays: translation ±2 px with probability 0.50, then mild darkening with probability 0.25 and brightness factor 0.90–1.00, then RGB → L → RGB with probability 0.10. All three random streams stay unchanged. Each new model starts with random weights; parent checkpoints are comparison evidence.

Clean evaluation disables dropout and random augmentation. **G2, E6, the saved 04ac models and the new models are all scored against the same new labels**, using IEEE FP32. Old run artifacts are verified against their original teacher labels first. Do not compare an old teacher-label score directly with a new name-label score.

This is a label sensitivity experiment proposed after looking at development errors. It is not an independent blind test or an accepted submission model.

In [3]:
from fashion.data.gender_name_truth import build_gender_name_truth_variant
from fashion.train.task3_gender_name_truth import (
    check_gender_name_truth_sources,
    run_gender_name_truth_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
DROPOUT_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030/gender"
DARKENING_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening/gender"
GRAYSCALE_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening_grayscale_010/gender"
)
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"

summary = build_gender_name_truth_variant(REPO_DIR)
print("Changed gender labels:", summary["changed_labels"])
print("Unclear names kept:", summary["no_cue_rows"] + summary["multiple_cue_rows"])
print("Fold label changes:", summary["folds"])
sources, classes, spec, evidence = check_gender_name_truth_sources(
    g2_directory=G2_DIR, e6_directory=E6_DIR, dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR, grayscale_directory=GRAYSCALE_DIR,
    source_registry_path=DRIVE_REGISTRY, precision_directory=PRECISION_DIR, root=REPO_DIR,
)
assert spec.classifier_dropout == 0.30
assert spec.to_dict()["grayscale_probability"] == 0.10
print("Verified source runs:", {name: len(runs) for name, runs in sources.items()})
print("Direct grayscale parents:", {fold: run["run_id"] for fold, run in sources["Gray10"].items()})
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")

Changed gender labels: 350
Unclear names kept: 1320
Fold label changes: [{'fold': 0, 'validation_rows': 6553, 'changed_validation_labels': 73, 'changed_training_labels': 277}, {'fold': 1, 'validation_rows': 6556, 'changed_validation_labels': 85, 'changed_training_labels': 265}, {'fold': 2, 'validation_rows': 6553, 'changed_validation_labels': 58, 'changed_training_labels': 292}, {'fold': 3, 'validation_rows': 6554, 'changed_validation_labels': 50, 'changed_training_labels': 300}, {'fold': 4, 'validation_rows': 6557, 'changed_validation_labels': 84, 'changed_training_labels': 266}]
Verified source runs: {'G2': 5, 'E6': 5, 'Drop30': 2, 'Drop30Dark': 2, 'Gray10': 2}
Direct grayscale parents: {0: 't3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f0_s2753_eb37119b7e68_20260905T144455Ze47ef6', 4: 't3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f4_s2753_eb37119b7e68_20260905T145521Z7d0a34'}
Frozen recipe: {'name': 'gender_name_truth_dropout_030_gr

## 4. Agreed screen rules

All rules must pass, using the same full-FP32 evaluation and the same name-truth labels for each model:

- Pooled validation macro-F1 falls by at most **0.030 versus matched G2**. The paired whole-family bootstrap 95% lower bound for the difference must be **at least −0.030**. Neither fold may lose more than 0.030. Use 10,000 draws within folds, seed 2753.
- The mean clean training–validation F1 gap falls by at least **0.050**. **Both folds' gaps must shrink.** Use clean evaluation-mode training scores from each finished checkpoint, not online augmented training scores.
- Preserve the existing stricter class guard: no pooled class loses more than **0.020 F1**. Thus the overall 0.030 allowance does not override a class failure. NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2; these measure the quality of model confidence.
- Preserve corruption guards versus matched E6: the translation-induced F1 change improves by at least 0.030; every other standard corruption, including darkening, worsens by at most 0.020. Each corrupted score is measured relative to that model's clean score.
- Exactly **390,181 parameters** and peak allocated GPU memory **strictly below 3,000,000,000 bytes**. Training time and latency are reported without speed caps. A memory failure stops before another fold begins.

The thresholds are derived from the new matched IEEE reference scores; do not substitute the rounded historical values. A pass means this screen met the chosen trade-off, not that the model is accepted or more accurate.


In [4]:
result = run_gender_name_truth_screen(
    g2_directory=G2_DIR, e6_directory=E6_DIR, dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR, grayscale_directory=GRAYSCALE_DIR,
    source_registry_path=DRIVE_REGISTRY, precision_directory=PRECISION_DIR,
    output_root=DRIVE_TASK_DIR, registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,), root=REPO_DIR,
)
print("Screen:", result["status"])
print("Label basis:", result.get("comparison_label_basis"))
for row in result.get("folds", []):
    print("Fold", row["fold"], "train F1:", row["candidate_train_f1"],
          "validation F1:", row["candidate_validation_f1"], "gap:", row["candidate_gap"])
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])
if "incremental_comparison" in result:
    incremental = result["incremental_comparison"]
    print("Direct grayscale parents:", result["direct_parent_run_ids"])
    print("F1 change versus Gray10 on the SAME new labels:", incremental["validation_delta"])
    print("Paired 95% interval:", incremental["validation_interval"])
    print("Class F1 changes:", incremental["class_f1_delta"])
    print("Induced corruption changes:", incremental["mean_induced_change_delta"])

IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f0_s2753_cb072542dbdc_20260904T135102Z6698e6
IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f4_s2753_cb072542dbdc_20260904T140011Z2211b1
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd
IEEE evaluation: t3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f0_s2753_eb37119b7e68_20260905T144455Ze47ef6
IEEE evaluation: t3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f4_s2753_eb37119b7e68_20260905T145521Z7d0a34
[task3] preparing target=gender fold=0: train=26,220 (before selection=26,220), validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_name_truth_dropout_030_grayscale_010_gender_smallcnngem3_f0_

## 5. Stop and review

Stop after folds 0 and 4. Do not auto-run folds 1–3, refit or open the held-out test. Every fit is recorded in `results/runs.csv` through the shared trainer. Matching completed runs are reused only after label, source, configuration, lineage and artifact checks. A memory failure stops before another fold starts.

Results are saved under `MyDrive/MLA2/task3/experiments/t3_gender_name_truth_dropout_030_grayscale_010/gender`:

- `screen_decision.json`: the 19 rules versus G2/E6 on the new labels, plus direct grayscale parent IDs.
- `incremental_comparison.json`: the effect of retraining versus the completed 04ac models, all scored on the same new labels. Existing `dropout_*` fields refer to these Gray10 parents.
- `clean_gap_comparison.csv` and `ieee_oof_predictions.csv`: clean scores and validation probabilities.
- `label_basis_comparison.csv` and `original_label_diagnostic.json`: score the same saved predictions against both name-truth and original teacher labels. This uses no extra GPU pass and adds no acceptance gates.
- `source_audit.json`, `label_variant/` and `comparison_name_truth_ieee/`: source and label hashes, label evidence, and separate matched model evaluations.

Read Boys/Girls class scores, raw grayscale scores and clean scores together. A smaller corruption drop can come from worse clean scores. A lower training score alone is not success. Keep this label experiment separate from the original teacher-label results in the report. Passing this screen does not by itself accept a final model.